# 01 — Explore Fruits-360 Dataset

**Fruvia AI** — Dataset exploration notebook.

This notebook is designed to run on **Google Colab**. It downloads the
[Fruits-360 Original Size](https://www.kaggle.com/datasets/moltean/fruits)
dataset via the Kaggle API and produces a comprehensive exploration report.

### What this notebook does

1. Download Fruits-360 from Kaggle into `/content/fruits360`
2. Discover all classes and count images per class
3. Print the full class list with image counts
4. Visualize the class distribution (bar chart)
5. Show sample images from random classes
6. Detect corrupt / unreadable images
7. Export a CSV summary of per-class statistics

### Prerequisites

- Store your **Kaggle API token** in Colab Secrets as `KAGGLE_USERNAME` and `KAGGLE_KEY`
  (Settings ⚙️ → Secrets → Add)
- No GPU required — CPU runtime is sufficient

## 1. Setup & Dependencies

In [ ]:
# Install any missing packages
!pip install -q kagglehub pyyaml Pillow matplotlib pandas

In [ ]:
import os
import random
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# Reproducibility
random.seed(42)

## 2. Download Fruits-360 from Kaggle

In [ ]:
from google.colab import userdata

# Read Kaggle credentials from Colab Secrets
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

import kagglehub

# Download the Fruits-360 Original Size dataset
dataset_path = kagglehub.dataset_download("moltean/fruits")
print(f"Dataset downloaded to: {dataset_path}")

# The dataset has Training/ and Test/ folders inside
DATA_DIR = Path(dataset_path)
print(f"Contents: {sorted([p.name for p in DATA_DIR.iterdir()])}")

## 3. Discover Classes & Count Images

In [ ]:
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}


def scan_split(split_dir: Path) -> dict[str, list[Path]]:
    """Scan a split directory (Training/Test) and return {class_name: [image_paths]}."""
    class_images: dict[str, list[Path]] = defaultdict(list)
    if not split_dir.exists():
        return class_images
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        for img_path in sorted(class_dir.iterdir()):
            if img_path.is_file() and img_path.suffix.lower() in SUPPORTED_EXTENSIONS:
                class_images[class_dir.name].append(img_path)
    return class_images


# Scan both Training and Test splits
splits = {}
for split_name in ["Training", "Test"]:
    split_dir = DATA_DIR / split_name
    if split_dir.exists():
        splits[split_name] = scan_split(split_dir)
        print(f"{split_name}: {len(splits[split_name])} classes")
    else:
        print(f"{split_name}: directory not found — checking alternatives...")
        # Some versions use lowercase or different names
        for alt in [split_name.lower(), "train", "test"]:
            alt_dir = DATA_DIR / alt
            if alt_dir.exists():
                splits[split_name] = scan_split(alt_dir)
                print(f"  Found as '{alt}': {len(splits[split_name])} classes")
                break

# Merge all classes across splits
all_classes: set[str] = set()
for split_data in splits.values():
    all_classes.update(split_data.keys())

print(f"\nTotal unique classes across all splits: {len(all_classes)}")

## 4. Full Class List with Image Counts

In [ ]:
# Build a summary table: class_name → count per split → total
rows = []
for cls in sorted(all_classes):
    row = {"class_name": cls}
    total = 0
    for split_name, split_data in sorted(splits.items()):
        count = len(split_data.get(cls, []))
        row[split_name] = count
        total += count
    row["total"] = total
    rows.append(row)

df_classes = pd.DataFrame(rows)
print(f"{'Class':<35} {'Training':>10} {'Test':>10} {'Total':>10}")
print("=" * 70)
for _, r in df_classes.iterrows():
    print(
        f"{r['class_name']:<35} {r.get('Training', 0):>10} {r.get('Test', 0):>10} {r['total']:>10}"
    )
print("=" * 70)
print(
    f"{'TOTAL':<35} "
    f"{df_classes.get('Training', pd.Series([0])).sum():>10} "
    f"{df_classes.get('Test', pd.Series([0])).sum():>10} "
    f"{df_classes['total'].sum():>10}"
)

## 5. Distribution Chart

In [ ]:
# Bar chart of total images per class (sorted descending)
df_sorted = df_classes.sort_values("total", ascending=True)

fig, ax = plt.subplots(figsize=(12, max(8, len(df_sorted) * 0.25)))
ax.barh(df_sorted["class_name"], df_sorted["total"], color="#4A90D9")
ax.set_xlabel("Number of Images")
ax.set_title("Fruits-360: Images per Class")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Add count labels
for i, (_, row) in enumerate(df_sorted.iterrows()):
    ax.text(row["total"] + 2, i, str(row["total"]), va="center", fontsize=7)

plt.tight_layout()
plt.show()

## 6. Sample Images

In [ ]:
# Show a grid of sample images from random classes
NUM_CLASSES_TO_SHOW = 12
IMAGES_PER_CLASS = 4

# Pick a split that has data (prefer Training)
show_split = "Training" if "Training" in splits else list(splits.keys())[0]
show_data = splits[show_split]

# Sample random classes
sample_classes = random.sample(
    sorted(show_data.keys()),
    min(NUM_CLASSES_TO_SHOW, len(show_data)),
)

fig, axes = plt.subplots(
    len(sample_classes),
    IMAGES_PER_CLASS,
    figsize=(IMAGES_PER_CLASS * 2.5, len(sample_classes) * 2.5),
)
if len(sample_classes) == 1:
    axes = [axes]

for row_idx, cls in enumerate(sample_classes):
    images = show_data[cls]
    sample_imgs = random.sample(images, min(IMAGES_PER_CLASS, len(images)))
    for col_idx in range(IMAGES_PER_CLASS):
        ax = axes[row_idx][col_idx] if IMAGES_PER_CLASS > 1 else axes[row_idx]
        if col_idx < len(sample_imgs):
            try:
                img = Image.open(sample_imgs[col_idx])
                ax.imshow(img)
            except Exception:
                ax.text(0.5, 0.5, "ERROR", ha="center", va="center", transform=ax.transAxes)
        ax.set_xticks([])
        ax.set_yticks([])
        if col_idx == 0:
            ax.set_ylabel(cls, fontsize=8, rotation=0, labelpad=80, va="center")

fig.suptitle(f"Sample Images from {show_split} Split", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. Detect Corrupt / Unreadable Images

In [ ]:
def check_image(filepath: Path) -> tuple[bool, str]:
    """Verify an image can be opened and decoded by Pillow."""
    try:
        with Image.open(filepath) as img:
            img.verify()
        return True, "ok"
    except Exception as e:
        return False, str(e)


corrupt_images: list[dict] = []
total_checked = 0

for split_name, split_data in splits.items():
    for cls, images in split_data.items():
        for img_path in images:
            total_checked += 1
            is_valid, reason = check_image(img_path)
            if not is_valid:
                corrupt_images.append(
                    {
                        "split": split_name,
                        "class": cls,
                        "filename": img_path.name,
                        "path": str(img_path),
                        "error": reason,
                    }
                )

print(f"Total images checked: {total_checked}")
print(f"Corrupt / unreadable: {len(corrupt_images)}")

if corrupt_images:
    print("\nCorrupt images:")
    for ci in corrupt_images:
        print(f"  [{ci['split']}] {ci['class']}/{ci['filename']}: {ci['error']}")

## 8. Export CSV Summary

In [ ]:
# Save the class distribution table as CSV
OUTPUT_CSV = Path("/content/fruits360_stats.csv")

df_classes.to_csv(OUTPUT_CSV, index=False)
print(f"CSV summary saved to {OUTPUT_CSV}")
print(f"  {len(df_classes)} classes, {df_classes['total'].sum()} total images")

# Also save corrupt image report if any
if corrupt_images:
    corrupt_csv = Path("/content/fruits360_corrupt.csv")
    pd.DataFrame(corrupt_images).to_csv(corrupt_csv, index=False)
    print(f"Corrupt image report saved to {corrupt_csv}")

## 9. (Optional) Save to Google Drive

In [ ]:
# Uncomment to mount Google Drive and copy the CSV there
# from google.colab import drive
# drive.mount("/content/drive")
#
# import shutil
# DRIVE_DIR = Path("/content/drive/MyDrive/fruvia-ai/data")
# DRIVE_DIR.mkdir(parents=True, exist_ok=True)
# shutil.copy2(OUTPUT_CSV, DRIVE_DIR / OUTPUT_CSV.name)
# print(f"Copied to {DRIVE_DIR / OUTPUT_CSV.name}")

---

## Summary

| Metric | Value |
|--------|-------|
| Total classes | (see above) |
| Total images | (see above) |
| Corrupt images | (see above) |

**Next step:** Run `02_prepare_dataset.ipynb` to create the classification and retrieval manifests.